## Service Health Prediction

Objective:
Predict the operational health of every OpenStack service using
aggregated service metrics generated in the Gold Layer.

Output Classes:
• Healthy
• Degrading
• Critical

Models:
• Logistic Regression
• Decision Tree
• Random Forest

Experiment Tracking:
• MLflow

In [0]:
## Load dependencies
# %run ./00_Project_Setup

In [0]:
%run ./06_ML_Utilities

In [0]:
## Read Gold Dataset
silver_df = spark.table(
    "`log-analytics`.silver.silver_logs"
)

print("="*60)
print("SILVER DATASET")
print("="*60)

print("Rows    :", silver_df.count())
print("Columns :", len(silver_df.columns))

display(silver_df)

In [0]:
## Validate Dataset
silver_df.printSchema()

display(
    silver_df.describe()
)

In [0]:
## Checking for NUll Values
from pyspark.sql import functions as F

null_summary = (
    silver_df.select([
        F.count(
            F.when(F.col(c).isNull(), c)
        ).alias(c)

        for c in silver_df.columns
    ])
)

display(null_summary)

In [0]:
## Droping Columns that will never help any ML Model
drop_columns = [
    "event_id",
    "timestamp",
    "message",
    "kafka_partition",
    "kafka_offset",
    "kafka_timestamp",
    "ingestion_timestamp"
]

service_health_df = silver_df.drop(*drop_columns)

In [0]:
## Handling High NULL Columns
fill_unkown = [
    "instance_id",
    "request_id",
    "user_id",
    "project_id",
    "client_ip",
    "http_method",
    "http_path"
]

service_health_df = service_health_df.fillna("UNKOWN", subset = fill_unkown)
service_health_df = service_health_df.fillna(-1, subset = "status_code")
service_health_df = service_health_df.fillna(0.0, subset = "response_time")

In [0]:
## Create Time Features
from pyspark.sql import functions as F

service_health_df = (
    service_health_df
    .withColumn("hour", F.hour("event_timestamp"))
    .withColumn("day_of_week", F.dayofweek("event_timestamp"))
    .withColumn("month", F.month("event_timestamp"))
    .withColumn("is_weekend",
                F.when(F.dayofweek("event_timestamp").isin([1,7]),1).otherwise(0))
)

display(service_health_df)

In [0]:
## Save the ML Dataset
service_health_df.printSchema()

display(
    service_health_df.describe()
)

In [0]:
## Creating Target Column "Service Health"
from pyspark.sql import functions as F

service_health_df = (
    service_health_df
    .withColumn(
        "service_health",
        F.when(
            (F.col("log_level") == "CRITICAL") |
            ((F.col("log_level") == "ERROR") & (F.col("response_time") >= 0.6)) |
            (F.col("severity_score") >= 4) |
            (F.col("status_code") >= 500),
            "Critical"
        )
        .when(
            (F.col("log_level").isin("ERROR","WARNING")) |
            (F.col("is_anomaly") == 1) |
            (F.col("response_time") > 0.35) |
            (F.col("status_code") >= 400),
            "Degrading"
        )
        .otherwise("Healthy")
    )
)

In [0]:
display(
    service_health_df.groupBy("service_health").count()
)

In [0]:
## Encoding Target Column
from pyspark.ml.feature import StringIndexer

health_indexer = StringIndexer(
    inputCol = "service_health",
    outputCol = "label",
    handleInvalid = "keep"
)

health_indexer_model = health_indexer.fit(service_health_df)

service_health_df = health_indexer_model.transform(service_health_df)

In [0]:
display(
    service_health_df
    .select("service_health", "label")
    .distinct()
    .orderBy("label")
)

In [0]:
## Feature Selection
feature_cols = [
    "service",
    "log_file",
    "log_level",
    "http_method",
    "status_code",
    "response_time",
    "event_hour",
    "event_month",
    "event_year",
    "day_of_week",
    "hour",
    "month",
    "is_weekend",
    "severity_score",
    "response_category",
    "instance_exists"
]

In [0]:
# Filter out non-existent columns from feature_cols
available_cols = service_health_df.columns
valid_feature_cols = [col for col in feature_cols if col in available_cols]

health_model_df = service_health_df.select(
    *valid_feature_cols,
    "label"
)

display(health_model_df.limit(10))

In [0]:
from pyspark.sql import functions as F

service_health_df = service_health_df.withColumn(
    "health_model_label",
    F.when(F.col("service_health") == "Healthy", "Healthy")
     .when(F.col("service_health") == "Degrading", "Degrading")
     .otherwise(None)
)

In [0]:
health_model_df = service_health_df.filter(
    F.col("health_model_label").isNotNull()
)

In [0]:
display(
    health_model_df.groupBy("health_model_label").count()
)

In [0]:
## Check Class Distribution
train_df, test_df = health_model_df.randomSplit(
    [0.8, 0.2],
    seed = 42
)

display(
    train_df.groupBy("label")
            .count()
            .orderBy("label")
)

In [0]:
## Add Class Weights
from pyspark.sql import functions as F

class_counts = train_df.groupBy("label").count().collect()

total = train_df.count()
num_classes = len(class_counts)

weights = {
    row["label"] : total / (num_classes * row["count"])
    for row in class_counts
}

print(weights)

In [0]:
train_df = train_df.withColumn(
    "class_weight",
    F.when(F.col("label") == 0, weights[0.0])
     .when(F.col("label") == 1, weights[1.0])
)

display(
    train_df.groupBy("label", "class_weight").count()
)

In [0]:
## Build the Feature Processing pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

categorical_cols = [
    "service",
    "log_file",
    "log_level",
    "http_method",
    "response_category"
]

numerical_cols = [
    "status_code",
    "response_time",
    "event_hour",
    "event_month",
    "event_year",
    "day_of_week",
    "hour",
    "month",
    "is_weekend",
    "severity_score"
]

indexers = [
    StringIndexer(
        inputCol = col,
        outputCol = f"{col}_index",
        handleInvalid = "keep"
    )
    for col in categorical_cols
]

encoders = [
    OneHotEncoder(
        inputCol = f"{col}_index",
        outputCol = f"{col}_encoded"
    )
    for col in categorical_cols
]

assembler = VectorAssembler(
    inputCols = numerical_cols + [f"{col}_encoded" for col in categorical_cols],
    outputCol = "features"
)

In [0]:
import gc

gc.collect()

In [0]:
## Build and fir the feature pipeline
from pyspark.ml import Pipeline

health_pipeline = Pipeline(
    stages = indexers + encoders + [assembler]
)

health_pipeline_model = health_pipeline.fit(train_df)

In [0]:
train_processed = health_pipeline_model.transform(train_df)
test_processed = health_pipeline_model.transform(test_df)

In [0]:
display(
    train_processed.select(
        "label",
        "class_weight",
        "features"
    ).limit(5)
)

In [0]:
## Lets train logistic regression
from pyspark.ml.classification import (
    LogisticRegression,
    DecisionTreeClassifier,
    RandomForestClassifier
)

from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

lr = LogisticRegression(
    featuresCol = "features",
    labelCol = "label",
    weightCol = "class_weight"
)

dt = DecisionTreeClassifier(
    featuresCol = "features",
    labelCol = "label",
    weightCol = "class_weight",
    seed = 42
)

rf = RandomForestClassifier(
    featuresCol = "features",
    labelCol = "label",
    weightCol = "class_weight",
    seed = 42
)


In [0]:
## Define Grids
lr_grid = (
    ParamGridBuilder()
    .addGrid(lr.regParam, [0.001, 0.01, 0.1])
    .addGrid(lr.elasticNetParam, [0.0, 0.5])
    .build()
)

dt_grid = (
    ParamGridBuilder()
    .addGrid(dt.maxDepth, [5,10,15])
    .addGrid(dt.minInstancesPerNode, [1,5,10])
    .build()
)

rf_grid = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [50,100])
    .addGrid(rf.maxDepth, [5,10])
    .build()
)

In [0]:
## Performing Cross Validation
from pyspark.ml.tuning import CrossValidator
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(
    labelCol = "label",
    predictionCol = "prediction",
    metricName = "f1"
)

lr_cv = CrossValidator(
    estimator = lr,
    estimatorParamMaps = lr_grid,
    evaluator = evaluator,
    numFolds = 3,
    seed = 42
)

dt_cv = CrossValidator(
    estimator = dt,
    estimatorParamMaps = dt_grid,
    evaluator = evaluator,
    numFolds = 3,
    seed = 42
)

rf_cv = CrossValidator(
    estimator = rf,
    estimatorParamMaps = rf_grid,
    evaluator = evaluator,
    numFolds = 3,
    seed = 42
)

In [0]:
## fitting data
import os

# Set temporary DataFrame storage path for cross-validation on serverless compute
os.environ['SPARKML_TEMP_DFS_PATH'] = '/Volumes/log-analytics/default/mlflow_tmp'

dbutils.fs.mkdirs("/Volumes/log-analytics/default/mlflow_tmp")

lr_cv_model = lr_cv.fit(train_processed)
dt_cv_model = dt_cv.fit(train_processed)
rf_dv_model = rf_cv.fit(train_processed)

In [0]:
lr_test = lr_cv_model.transform(test_processed)
dt_test = dt_cv_model.transform(test_processed)
rf_test = rf_dv_model.transform(test_processed)

In [0]:
## Claculating F1 for each
lr_f1 = evaluator.evaluate(lr_test)
dt_f1 = evaluator.evaluate(dt_test)
rf_f1 = evaluator.evaluate(rf_test)

print("Model Comparison")
print("="*40)
print(f"Logistic Regression F1 : {lr_f1:.4f}")
print(f"Decision Tree F1       : {dt_f1:.4f}")
print(f"Random Forest F1       : {rf_f1:.4f}")

In [0]:
## Save the best Model
model_scores = {
    "Logistic Regression": lr_f1,
    "Decision Tree": dt_f1,
    "Random Forest": rf_f1
}

best_model_name = max(model_scores, key=model_scores.get)

print("Best Model:", best_model_name)
print("F1 Score:", model_scores[best_model_name])

In [0]:
if best_model_name == "Logistic Regression":
    best_classifier = lr_cv_model.bestModel

elif best_model_name == "Decision Tree":
    best_classifier = dt_cv_model.bestModel

else:
    best_classifier = rf_cv_model.bestModel

In [0]:
from pyspark.ml import PipelineModel

final_health_model = PipelineModel(
    stages=health_pipeline_model.stages + [best_classifier]
)

In [0]:
model_path = "/Volumes/log-analytics/bronze/key_volume/models/service_health_model"

final_health_model.write().overwrite().save(model_path)

print("Service Health model saved successfully.")